In [ ]:

# Artistic Portrait Enhancement using
# Bilateral Filtering + MSRCR + LBP + VGG19
# ============================================================

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import os
import cv2
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm import tqdm

from skimage.feature import local_binary_pattern
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms


import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import label_binarize
# plotting settings
plt.rcParams["figure.figsize"] = (12, 9)
plt.rcParams["figure.dpi"] = 800          # <-- added DPI
#plt.rcParams["font.family"] = "Liberation Serif"
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.size"] = 20
plt.rcParams["font.weight"] = "bold"
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelweight"] = "bold"
plt.rcParams["axes.labelsize"] = 20

plt.rcParams["axes.grid"] = False
plt.rcParams["xtick.labelsize"] = 20
plt.rcParams["ytick.labelsize"] = 20
# ============================================================
# 2. CONFIGURATION
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATASET_PATH = "dataset/*.jpg"
STYLE_IMAGE_PATH = "style/style.jpg"

IMAGE_SIZE = 512

CONTENT_WEIGHT = 1e5
STYLE_WEIGHT = 1e10

NUM_ITERATIONS = 300

LBP_RADIUS = 1
LBP_POINTS = 8 * LBP_RADIUS

# ============================================================
# 3. IMAGE TRANSFORM
# ============================================================

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor()
])


# ============================================================
# 4. LOAD IMAGE
# ============================================================

def load_image(path):

    image = Image.open(path).convert("RGB")

    tensor = transform(image).unsqueeze(0)

    return tensor.to(DEVICE)


# ============================================================
# 5. BILATERAL FILTERING
# ============================================================

def bilateral_filtering(image):

    filtered = cv2.bilateralFilter(
        image,
        d=9,
        sigmaColor=75,
        sigmaSpace=75
    )

    return filtered


# ============================================================
# 6. SINGLE SCALE RETINEX
# ============================================================

def single_scale_retinex(img, sigma):

    blur = cv2.GaussianBlur(img, (0,0), sigma)

    retinex = np.log10(img + 1.0) - np.log10(blur + 1.0)

    return retinex


# ============================================================
# 7. MULTI SCALE RETINEX
# ============================================================

def multi_scale_retinex(img, sigma_list):

    retinex = np.zeros_like(img)

    for sigma in sigma_list:

        retinex += single_scale_retinex(img, sigma)

    retinex = retinex / len(sigma_list)

    return retinex


# ============================================================
# 8. COLOR RESTORATION
# ============================================================

def color_restoration(img, alpha=125.0, beta=46.0):

    img_sum = np.sum(img, axis=2, keepdims=True)

    color = beta * (
        np.log10(alpha * img) - np.log10(img_sum)
    )

    return color


# ============================================================
# 9. SIMPLEST COLOR BALANCE
# ============================================================

def simplest_color_balance(img, low_clip=0.01, high_clip=0.99):

    total = img.shape[0] * img.shape[1]

    for i in range(img.shape[2]):

        unique, counts = np.unique(
            img[:,:,i],
            return_counts=True
        )

        current = 0

        low_val = unique[0]
        high_val = unique[-1]

        for u, c in zip(unique, counts):

            if float(current) / total < low_clip:
                low_val = u

            if float(current) / total < high_clip:
                high_val = u

            current += c

        img[:,:,i] = np.maximum(
            np.minimum(img[:,:,i], high_val),
            low_val
        )

    return img


# ============================================================
# 10. MSRCR
# ============================================================

def msrcr(img):

    img = np.float64(img) + 1.0

    retinex = multi_scale_retinex(
        img,
        [15, 80, 250]
    )

    color = color_restoration(img)

    msrcr_result = retinex * color

    for i in range(msrcr_result.shape[2]):

        msrcr_result[:,:,i] = (
            (msrcr_result[:,:,i] - np.min(msrcr_result[:,:,i]))
            /
            (np.max(msrcr_result[:,:,i]) - np.min(msrcr_result[:,:,i]))
            * 255
        )

    msrcr_result = np.uint8(
        np.minimum(
            np.maximum(msrcr_result, 0),
            255
        )
    )

    msrcr_result = simplest_color_balance(msrcr_result)

    return msrcr_result


# ============================================================
# 11. LBP FEATURE EXTRACTION
# ============================================================

def extract_lbp_features(image):

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    lbp = local_binary_pattern(
        gray,
        P=LBP_POINTS,
        R=LBP_RADIUS,
        method='uniform'
    )

    hist, _ = np.histogram(
        lbp.ravel(),
        bins=np.arange(0, LBP_POINTS + 3),
        range=(0, LBP_POINTS + 2)
    )

    hist = hist.astype("float")

    hist /= (hist.sum() + 1e-6)

    return lbp, hist


# ============================================================
# 12. VGG19 FEATURE MODEL
# ============================================================

class VGG19(nn.Module):

    def __init__(self):

        super(VGG19, self).__init__()

        self.model = models.vgg19(
            weights=models.VGG19_Weights.DEFAULT
        ).features

        self.layers = {
            '0': 'conv1_1',
            '5': 'conv2_1',
            '10': 'conv3_1',
            '19': 'conv4_1',
            '21': 'conv4_2',
            '28': 'conv5_1'
        }

    def forward(self, x):

        features = {}

        for name, layer in self.model._modules.items():

            x = layer(x)

            if name in self.layers:

                features[self.layers[name]] = x

        return features


vgg = VGG19().to(DEVICE).eval()


# ============================================================
# 13. GRAM MATRIX
# ============================================================

def gram_matrix(tensor):

    b, d, h, w = tensor.size()

    tensor = tensor.view(d, h*w)

    gram = torch.mm(
        tensor,
        tensor.t()
    )

    return gram


# ============================================================
# 14. STYLE TRANSFER TRAINING
# ============================================================

def style_transfer(content_path, style_path):

    content = load_image(content_path)

    style = load_image(style_path)

    generated = content.clone().requires_grad_(True)

    content_features = vgg(content)

    style_features = vgg(style)

    style_grams = {
        layer: gram_matrix(style_features[layer])
        for layer in style_features
    }

    optimizer = optim.Adam(
        [generated],
        lr=0.003
    )

    for step in range(NUM_ITERATIONS):

        generated_features = vgg(generated)

        content_loss = torch.mean(
            (
                generated_features['conv4_2']
                -
                content_features['conv4_2']
            ) ** 2
        )

        style_loss = 0

        for layer in style_grams:

            gen_feature = generated_features[layer]

            b, d, h, w = gen_feature.shape

            gen_gram = gram_matrix(gen_feature)

            style_gram = style_grams[layer]

            layer_loss = torch.mean(
                (gen_gram - style_gram) ** 2
            )

            style_loss += layer_loss / (d*h*w)

        total_loss = (
            CONTENT_WEIGHT * content_loss
            +
            STYLE_WEIGHT * style_loss
        )

        optimizer.zero_grad()

        total_loss.backward()

        optimizer.step()

        if step % 50 == 0:

            print(
                f"Iteration {step} | "
                f"Total Loss : {total_loss.item():.4f}"
            )

    output = generated.detach().cpu().squeeze()

    output = output.permute(1,2,0).numpy()

    output = np.clip(output, 0, 1)

    output = (output * 255).astype(np.uint8)

    return output


# ============================================================
# 15. METRIC CALCULATION
# ============================================================

def calculate_metrics(original, enhanced):

    original_gray = cv2.cvtColor(
        original,
        cv2.COLOR_BGR2GRAY
    )

    enhanced_gray = cv2.cvtColor(
        enhanced,
        cv2.COLOR_BGR2GRAY
    )

    psnr_value = psnr(
        original_gray,
        enhanced_gray
    )

    ssim_value = ssim(
        original_gray,
        enhanced_gray
    )

    error = np.mean(
        np.abs(
            original.astype(np.float32)
            -
            enhanced.astype(np.float32)
        )
    ) / 255.0

    return psnr_value, ssim_value, error


# ============================================================
# 16. MAIN TRAINING PIPELINE
# ============================================================

image_paths = glob.glob(DATASET_PATH)

psnr_values = []
ssim_values = []
error_values = []

for image_path in tqdm(image_paths):

    # --------------------------------------------------------
    # ORIGINAL IMAGE
    # --------------------------------------------------------

    original = cv2.imread(image_path)

    original = cv2.cvtColor(
        original,
        cv2.COLOR_BGR2RGB
    )

    original = cv2.resize(
        original,
        (IMAGE_SIZE, IMAGE_SIZE)
    )

    # --------------------------------------------------------
    # BILATERAL FILTERING
    # --------------------------------------------------------

    filtered = bilateral_filtering(original)

    # --------------------------------------------------------
    # MSRCR ENHANCEMENT
    # --------------------------------------------------------

    enhanced = msrcr(filtered)

    # --------------------------------------------------------
    # LBP FEATURE EXTRACTION
    # --------------------------------------------------------

    lbp_map, lbp_hist = extract_lbp_features(enhanced)

    # --------------------------------------------------------
    # SAVE TEMP IMAGE
    # --------------------------------------------------------

    temp_path = "temp.jpg"

    cv2.imwrite(
        temp_path,
        cv2.cvtColor(enhanced, cv2.COLOR_RGB2BGR)
    )

    # --------------------------------------------------------
    # VGG19 STYLE TRANSFER
    # --------------------------------------------------------

    stylized = style_transfer(
        temp_path,
        STYLE_IMAGE_PATH
    )

    # --------------------------------------------------------
    # METRICS
    # --------------------------------------------------------

    psnr_value, ssim_value, error_value = calculate_metrics(
        original,
        stylized
    )

    psnr_values.append(psnr_value)
    ssim_values.append(ssim_value)
    error_values.append(error_value)

    # ========================================================
    # PLOT 1 : BILATERAL FILTER COMPARISON
    # ========================================================

    fig, ax = plt.subplots(1,2, figsize=(12,5))

    ax[0].imshow(original)
    ax[0].set_title("Original Portrait")
    ax[0].axis("off")

    ax[1].imshow(filtered)
    ax[1].set_title("After Bilateral Filtering")
    ax[1].axis("off")

    plt.tight_layout()
    plt.show()

    # ========================================================
    # PLOT 2 : HISTOGRAM DISTRIBUTION
    # ========================================================

    fig, ax = plt.subplots(1,2, figsize=(12,5))

    ax[0].hist(original.ravel(), bins=256)
    ax[0].set_title("Original Histogram")

    ax[1].hist(enhanced.ravel(), bins=256)
    ax[1].set_title("Enhanced Histogram")

    plt.tight_layout()
    plt.show()

    # ========================================================
    # PLOT 3 : CF CURVE
    # ========================================================

    original_gray = cv2.cvtColor(
        original,
        cv2.COLOR_RGB2GRAY
    )

    enhanced_gray = cv2.cvtColor(
        enhanced,
        cv2.COLOR_RGB2GRAY
    )

    fig, ax = plt.subplots(1,2, figsize=(12,5))

    ax[0].plot(
        cv2.calcHist(
            [original_gray],
            [0],
            None,
            [256],
            [0,256]
        )
    )

    ax[0].set_title("Original CF Curve")

    ax[1].plot(
        cv2.calcHist(
            [enhanced_gray],
            [0],
            None,
            [256],
            [0,256]
        )
    )

    ax[1].set_title("Enhanced CF Curve")

    plt.tight_layout()
    plt.show()

    # ========================================================
    # PLOT 4 : LBP FEATURE MAP
    # ========================================================

    plt.figure(figsize=(6,6))

    plt.imshow(lbp_map, cmap='gray')

    plt.title("LBP Feature Map")

    plt.axis("off")

    plt.show()

    # ========================================================
    # PLOT 5 : INPUT VS ENHANCED
    # ========================================================

    fig, ax = plt.subplots(1,2, figsize=(14,6))

    ax[0].imshow(original)
    ax[0].set_title("Input Portrait")
    ax[0].axis("off")

    ax[1].imshow(stylized)
    ax[1].set_title("Enhanced Portrait")
    ax[1].axis("off")

    plt.tight_layout()
    plt.show()

    # ========================================================
    # PLOT 6 : COMPLETE PIPELINE
    # ========================================================

    fig, ax = plt.subplots(1,4, figsize=(22,5))

    ax[0].imshow(original)
    ax[0].set_title("Original")

    ax[1].imshow(filtered)
    ax[1].set_title("Bilateral")

    ax[2].imshow(enhanced)
    ax[2].set_title("MSRCR")

    ax[3].imshow(stylized)
    ax[3].set_title("VGG19 Output")

    for a in ax:
        a.axis("off")

    plt.tight_layout()
    plt.show()

    # ========================================================
    # PLOT 7 : LBP HISTOGRAM
    # ========================================================

    plt.figure(figsize=(8,5))

    plt.hist(
        lbp_map.ravel(),
        bins=30
    )

    plt.title("LBP Histogram Distribution")

    plt.xlabel("LBP Pattern")

    plt.ylabel("Frequency")

    plt.show()


# ============================================================
# 17. FINAL PERFORMANCE GRAPH
# ============================================================

plt.figure(figsize=(10,6))

plt.plot(
    psnr_values,
    marker='o',
    label='PSNR'
)

plt.plot(
    ssim_values,
    marker='s',
    label='SSIM'
)

plt.plot(
    error_values,
    marker='^',
    label='MSRCR Error'
)

plt.xlabel("Image Index")

plt.ylabel("Metric Value")

plt.title("Performance Evaluation")

plt.legend()

plt.grid(True)

plt.show()


# ============================================================
# 18. AVERAGE RESULTS
# ============================================================

avg_psnr = np.mean(psnr_values)

avg_ssim = np.mean(ssim_values)

avg_error = np.mean(error_values)

print("\n==============================")
print("AVERAGE PERFORMANCE")
print("==============================")
print(f"Average PSNR        : {avg_psnr:.2f} dB")
print(f"Average SSIM        : {avg_ssim:.4f}")
print(f"Average MSRCR Error : {avg_error:.4f}")
print("==============================")


# ============================================================
# 19. FINAL BAR CHART
# ============================================================

metrics = ['PSNR', 'SSIM', 'MSRCR Error']

values = [
    avg_psnr,
    avg_ssim,
    avg_error
]

plt.figure(figsize=(8,5))

bars = plt.bar(metrics, values)

for bar in bars:

    height = bar.get_height()

    plt.text(
        bar.get_x() + bar.get_width()/2,
        height,
        f'{height:.2f}',
        ha='center'
    )

plt.title("Average Quantitative Results")

plt.ylabel("Value")

plt.show()

# ============================================================
# END OF COMPLETE ORIGINAL IMPLEMENTATION
# ============================================================